In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)

print("Step 1: Deriving Telematics Features from CAN Signals...")
# 1. Simulate Raw CAN Data Streams (Speed & RPM Proxies)
# In a real scenario, you decode payload bytes from specific CAN IDs (e.g., ID 0x316 for RPM).
# Here, we generate raw CAN-level signal distributions[cite: 42, 253].
n_can_messages = 100000

# Normal CAN signals
can_speed_normal = np.random.normal(60, 15, n_can_messages)  # km/h proxy
can_rpm_normal = np.random.normal(2000, 400, n_can_messages) # RPM proxy
can_fuel_normal = np.random.normal(8, 1.5, n_can_messages)   # Fuel consumption rate proxy

# 2. Aggregate into Telematics "Trip Summaries" [cite: 254]
# Telematics devices don't send every CAN message to the cloud; they send aggregated windows (e.g., every 60 seconds).
window_size = 100 # Aggregate every 100 CAN messages into 1 telematics update
n_windows = n_can_messages // window_size

derived_telematics = []
for i in range(n_windows):
    start_idx = i * window_size
    end_idx = start_idx + window_size
    
    # Calculate derived features from the CAN window [cite: 252, 253]
    derived_telematics.append({
        'derived_speed_mean': np.mean(can_speed_normal[start_idx:end_idx]),
        'derived_speed_max': np.max(can_speed_normal[start_idx:end_idx]),
        'derived_rpm_mean': np.mean(can_rpm_normal[start_idx:end_idx]),
        'derived_fuel_rate': np.mean(can_fuel_normal[start_idx:end_idx]),
        'location_cluster': np.random.randint(0, 5), # Synthetic location proxy (e.g., 0-4 are normal commute zones) [cite: 255]
        'is_anomaly': 0
    })

df_normal_fleet = pd.DataFrame(derived_telematics)

print("Step 2: Generating Synthetic Fleet Profiles for Anomalous Behavior...")
# 3. Simulate Attack Scenarios (Theft / Remote Takeover) [cite: 43, 82]
# Anomalous profile: High RPMs without matching speed (gear spoofing/neutral revving), deviated locations.
n_anomaly_windows = int(n_windows * 0.05) # 5% of data is anomalous
df_anomaly_fleet = pd.DataFrame({
    'derived_speed_mean': np.random.normal(40, 20, n_anomaly_windows),
    'derived_speed_max': np.random.normal(80, 25, n_anomaly_windows),
    'derived_rpm_mean': np.random.normal(5000, 500, n_anomaly_windows), # Unusually high RPM [cite: 43]
    'derived_fuel_rate': np.random.normal(15, 3, n_anomaly_windows),
    'location_cluster': np.random.randint(5, 10, n_anomaly_windows),    # Unrecognized location clusters [cite: 75]
    'is_anomaly': 1
})

# Combine into our final Telematics Dataset
telematics_df = pd.concat([df_normal_fleet, df_anomaly_fleet]).sample(frac=1).reset_index(drop=True)

print(f"Total telematics updates generated: {len(telematics_df)}")
print(f"Anomalies (Attacks): {telematics_df['is_anomaly'].sum()}")

print("\nStep 3: Preparing Data & Training the Autoencoder Baseline...")
features = ['derived_speed_mean', 'derived_speed_max', 'derived_rpm_mean', 'derived_fuel_rate', 'location_cluster']
X = telematics_df[features]
y = telematics_df['is_anomaly']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Autoencoder ONLY on normal fleet data [cite: 78, 79]
X_train_normal = X_train[y_train == 0]

scaler = StandardScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled = scaler.transform(X_test)

# Build Autoencoder
input_dim = X_train_normal_scaled.shape[1]
autoencoder = models.Sequential([
    layers.Dense(16, activation='relu', input_shape=(input_dim,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(4, activation='relu'), # Bottleneck
    layers.Dense(8, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(input_dim, activation='linear')
])

autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(X_train_normal_scaled, X_train_normal_scaled, epochs=20, batch_size=32, validation_split=0.1, verbose=0)
print("Autoencoder training complete.")

print("\nStep 4: Evaluating Anomaly Scenarios...")
reconstructions = autoencoder.predict(X_test_scaled)
mse = np.mean(np.power(X_test_scaled - reconstructions, 2), axis=1)

# Define "normal" behavior envelope threshold (95th percentile) [cite: 79]
train_reconstructions = autoencoder.predict(X_train_normal_scaled)
train_mse = np.mean(np.power(X_train_normal_scaled - train_reconstructions, 2), axis=1)
threshold = np.percentile(train_mse, 95)

y_pred = (mse > threshold).astype(int)

print(f"\nAnomaly Threshold (MSE): {threshold:.4f}")
print("\n=== Telematics Behavioral Detection Performance ===")
print(classification_report(y_test, y_pred, target_names=['Normal Behavior', 'Anomalous/Abuse']))

Step 1: Deriving Telematics Features from CAN Signals...
Step 2: Generating Synthetic Fleet Profiles for Anomalous Behavior...
Total telematics updates generated: 1050
Anomalies (Attacks): 50

Step 3: Preparing Data & Training the Autoencoder Baseline...


d:\Assignments\automotive_ids_project\venv\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Autoencoder training complete.

Step 4: Evaluating Anomaly Scenarios...
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

Anomaly Threshold (MSE): 1.0250

=== Telematics Behavioral Detection Performance ===
                 precision    recall  f1-score   support

Normal Behavior       1.00      0.95      0.97       200
Anomalous/Abuse       0.50      1.00      0.67        10

       accuracy                           0.95       210
      macro avg       0.75      0.97      0.82       210
   weighted avg       0.98      0.95      0.96       210

